# Text Feature Engineering Assignment

> Dataset: Flipkart iPhone 15 reviews scraped into `flipkart_reviews.csv` using `fetch_flipkart_reviews.py`.

This notebook covers:
- real-world dataset loading
- text preprocessing
- vocabulary creation
- one-hot encoding, Bag of Words, and TF-IDF
- sparse matrix analysis
- sentiment classification with Logistic Regression

In [4]:
from collections import Counter
from pathlib import Path
import re
import string

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

DATA_PATH = Path("flipkart_reviews.csv")
NEGATION_WORDS = {"no", "not", "nor"}
STOP_WORDS = set(stopwords.words("english")) - NEGATION_WORDS
LEMMATIZER = WordNetLemmatizer()

pd.set_option("display.max_colwidth", 120)

In [2]:
reviews_df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {reviews_df.shape}")
reviews_df.head()

Dataset shape: (110, 8)


,review_title,review_text,sentiment,sort_order,page_number,product_variant,review_meta,source_url
0,Don't waste your money,Heating issues,negative,NEGATIVE_FIRST,1,Review for: Color Pink • Storage 128 GB,"Flipkart Customer , Mumbai 0 0 Verified Purchase · 5 days ago",https://www.flipkart.com/apple-iphone-15-black-128-gb/product-reviews/itm6ac6485515ae4?pid=MOBGTAGPTB3VS24W&lid=LSTM...
1,Don't waste your money,Spicker not work,negative,NEGATIVE_FIRST,1,Review for: Color Green • Storage 128 GB,"Anish Kumar , Kona 4 0 Verified Purchase · 3 months ago",https://www.flipkart.com/apple-iphone-15-black-128-gb/product-reviews/itm6ac6485515ae4?pid=MOBGTAGPTB3VS24W&lid=LSTM...
2,Horrible,Heating issue,negative,NEGATIVE_FIRST,1,Review for: Color Black • Storage 128 GB,"Tejas Wadekar , Pune 0 1 Verified Purchase · 1 month ago",https://www.flipkart.com/apple-iphone-15-black-128-gb/product-reviews/itm6ac6485515ae4?pid=MOBGTAGPTB3VS24W&lid=LSTM...
3,Horrible,"Screen is too small , performance below avarage No obsession to return Don't wast your money",negative,NEGATIVE_FIRST,1,Review for: Color Black • Storage 128 GB,"Flipkart Customer , Gautam Buddha Nagar 8 5 Verified Purchase · 2 months ago",https://www.flipkart.com/apple-iphone-15-black-128-gb/product-reviews/itm6ac6485515ae4?pid=MOBGTAGPTB3VS24W&lid=LSTM...
4,Horrible,Worst product ever,negative,NEGATIVE_FIRST,1,Review for: Color Green • Storage 128 GB,"Flipkart Customer , Prakasam 0 0 Verified Purchase · 8 days ago",https://www.flipkart.com/apple-iphone-15-black-128-gb/product-reviews/itm6ac6485515ae4?pid=MOBGTAGPTB3VS24W&lid=LSTM...


In [5]:
def preprocess_text(text: str, remove_stopwords: bool = True, lemmatize: bool = True) -> list[str]:
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = re.findall(r"[a-z0-9']+", text)
    if remove_stopwords:
        tokens = [token for token in tokens if token not in STOP_WORDS]
    if lemmatize:
        tokens = [LEMMATIZER.lemmatize(token) for token in tokens]
    return tokens

reviews_df["tokens"] = reviews_df["review_text"].apply(preprocess_text)
reviews_df["clean_text"] = reviews_df["tokens"].apply(" ".join)
reviews_df[["review_text", "clean_text", "sentiment"]].head(10)

,review_text,clean_text,sentiment
0,Heating issues,heating issue,negative
1,Spicker not work,spicker not work,negative
2,Heating issue,heating issue,negative
3,"Screen is too small , performance below avarage No obsession to return Don't wast your money",screen small performance avarage no obsession return dont wast money,negative
4,Worst product ever,worst product ever,negative
5,Not good,not good,negative
6,Build quality worst,build quality worst,negative
7,📸 bad,bad,negative
8,Battery issues within 6 months,battery issue within 6 month,negative
9,Wast,wast,negative


In [6]:
token_counter = Counter(token for tokens in reviews_df["tokens"] for token in tokens)
vocabulary = sorted(token_counter)
top_words_df = pd.DataFrame(token_counter.most_common(15), columns=["word", "frequency"])

print(f"Vocabulary size: {len(vocabulary)}")
top_words_df

Vocabulary size: 349


,word,frequency
0,camera,24
1,battery,23
2,not,19
3,iphone,19
4,phone,18
5,good,13
6,awesome,10
7,quality,8
8,flipkart,8
9,issue,7


In [7]:
ohe_vocabulary = vocabulary
ohe_matrix = pd.DataFrame(
    [{term: int(term in set(tokens)) for term in ohe_vocabulary} for tokens in reviews_df["tokens"]],
    dtype=int,
    index=reviews_df.index,
 )

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(reviews_df["clean_text"])

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(reviews_df["clean_text"])

feature_summary_df = pd.DataFrame(
    [
        {"method": "One Hot Encoding", "shape": ohe_matrix.shape, "non_zero_entries": int(ohe_matrix.to_numpy().sum())},
        {"method": "Bag of Words", "shape": bow_matrix.shape, "non_zero_entries": int(bow_matrix.nnz)},
        {"method": "TF-IDF", "shape": tfidf_matrix.shape, "non_zero_entries": int(tfidf_matrix.nnz)},
    ]
)

feature_summary_df

,method,shape,non_zero_entries
0,One Hot Encoding,"(110, 349)",665
1,Bag of Words,"(110, 344)",658
2,TF-IDF,"(110, 344)",658


In [8]:
def calculate_sparsity(matrix) -> float:
    if hasattr(matrix, "nnz"):
        total_values = matrix.shape[0] * matrix.shape[1]
        return 100 * (1 - (matrix.nnz / total_values))
    values = matrix.to_numpy()
    zero_count = np.count_nonzero(values == 0)
    return 100 * (zero_count / values.size)

sparsity_df = pd.DataFrame(
    [
        {"method": "One Hot Encoding", "shape": ohe_matrix.shape, "sparsity_percent": round(calculate_sparsity(ohe_matrix), 2)},
        {"method": "Bag of Words", "shape": bow_matrix.shape, "sparsity_percent": round(calculate_sparsity(bow_matrix), 2)},
        {"method": "TF-IDF", "shape": tfidf_matrix.shape, "sparsity_percent": round(calculate_sparsity(tfidf_matrix), 2)},
    ]
)

comparison_df = pd.DataFrame(
    [
        {
            "Method": "One Hot Encoding",
            "Representation": "Binary presence or absence of terms in each document",
            "Strengths": "Simple and interpretable",
            "Limitations": "Ignores term frequency and creates very wide matrices",
        },
        {
            "Method": "Bag of Words",
            "Representation": "Raw token counts per document",
            "Strengths": "Strong baseline for keyword-driven tasks",
            "Limitations": "Common words dominate without weighting",
        },
        {
            "Method": "TF-IDF",
            "Representation": "Term importance scaled by inverse document frequency",
            "Strengths": "Highlights discriminative words and downweights common terms",
            "Limitations": "Still ignores semantics and word order",
        },
    ]
)

display(sparsity_df)
comparison_df

,method,shape,sparsity_percent
0,One Hot Encoding,"(110, 349)",98.27
1,Bag of Words,"(110, 344)",98.26
2,TF-IDF,"(110, 344)",98.26


,Method,Representation,Strengths,Limitations
0,One Hot Encoding,Binary presence or absence of terms in each document,Simple and interpretable,Ignores term frequency and creates very wide matrices
1,Bag of Words,Raw token counts per document,Strong baseline for keyword-driven tasks,Common words dominate without weighting
2,TF-IDF,Term importance scaled by inverse document frequency,Highlights discriminative words and downweights common terms,Still ignores semantics and word order


In [9]:
tfidf_scores = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
tfidf_terms = np.array(tfidf_vectorizer.get_feature_names_out())
top_tfidf_df = pd.DataFrame(
    {"term": tfidf_terms, "mean_tfidf_score": tfidf_scores}
).sort_values("mean_tfidf_score", ascending=False).head(15).reset_index(drop=True)

print("Top TF-IDF terms are the words that are frequent in a smaller subset of reviews, while common words receive lower IDF weight.")
top_tfidf_df

Top TF-IDF terms are the words that are frequent in a smaller subset of reviews, while common words receive lower IDF weight.


,term,mean_tfidf_score
0,camera,0.059423
1,not,0.052886
2,battery,0.049512
3,awesome,0.047647
4,bad,0.044262
5,good,0.043985
6,phone,0.042331
7,iphone,0.033372
8,issue,0.029474
9,heating,0.029358


## Real-world Discussion

**Why Bag of Words fails for semantic meaning**

> Bag of Words treats words as independent counts. It cannot understand that phrases like `great camera` and `excellent camera` express similar meaning with different tokens, and it cannot capture negation or word order well.

**When to use Bag of Words and TF-IDF in industry**

> Bag of Words is useful for strong keyword-driven baselines, search indexing, and quick prototypes. TF-IDF is usually better when you want informative terms to matter more than very common terms, especially in document ranking, ticket routing, or lightweight text classifiers.

**Limitations of TF-IDF in real applications**

> TF-IDF still ignores context, sarcasm, synonyms, and sequence. It also requires a fixed vocabulary and can drift when product language changes over time.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    reviews_df["clean_text"],
    reviews_df["sentiment"],
    test_size=0.25,
    random_state=42,
    stratify=reviews_df["sentiment"],
)

bow_classifier = CountVectorizer()
X_train_bow = bow_classifier.fit_transform(X_train)
X_test_bow = bow_classifier.transform(X_test)

tfidf_classifier = TfidfVectorizer()
X_train_tfidf = tfidf_classifier.fit_transform(X_train)
X_test_tfidf = tfidf_classifier.transform(X_test)

bow_model = LogisticRegression(max_iter=2000)
bow_model.fit(X_train_bow, y_train)
bow_predictions = bow_model.predict(X_test_bow)

tfidf_model = LogisticRegression(max_iter=2000)
tfidf_model.fit(X_train_tfidf, y_train)
tfidf_predictions = tfidf_model.predict(X_test_tfidf)

model_results_df = pd.DataFrame(
    [
        {
            "feature_type": "Bag of Words",
            "accuracy": round(accuracy_score(y_test, bow_predictions), 4),
            "f1_score": round(f1_score(y_test, bow_predictions, pos_label="positive"), 4),
        },
        {
            "feature_type": "TF-IDF",
            "accuracy": round(accuracy_score(y_test, tfidf_predictions), 4),
            "f1_score": round(f1_score(y_test, tfidf_predictions, pos_label="positive"), 4),
        },
    ]
)

display(model_results_df)
print("Bag of Words report")
print(classification_report(y_test, bow_predictions))
print("TF-IDF report")
print(classification_report(y_test, tfidf_predictions))

,feature_type,accuracy,f1_score
0,Bag of Words,0.9643,0.9655
1,TF-IDF,0.9643,0.9655


Bag of Words report
              precision    recall  f1-score   support

    negative       1.00      0.93      0.96        14
    positive       0.93      1.00      0.97        14

    accuracy                           0.96        28
   macro avg       0.97      0.96      0.96        28
weighted avg       0.97      0.96      0.96        28

TF-IDF report
              precision    recall  f1-score   support

    negative       1.00      0.93      0.96        14
    positive       0.93      1.00      0.97        14

    accuracy                           0.96        28
   macro avg       0.97      0.96      0.96        28
weighted avg       0.97      0.96      0.96        28



## Final Observations

- The scraped review dataset is highly sparse after vectorization because most reviews use only a small fraction of the total vocabulary.
- One Hot Encoding is the simplest binary representation, but it loses count and importance information.
- Bag of Words is a practical baseline for classic ML text classification.
- TF-IDF usually gives cleaner signals for informative words because repeated common words are downweighted.
- The sentiment labels in this mini use case are derived from Flipkart positive-first and negative-first review sorting, so they are useful for demonstration but not a perfect gold-standard annotation set.